In [1]:
import pandas as pd
import numpy as np
import math
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
import gc

# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer


In [2]:
data = pd.read_csv('/mnt/dicoms/borja_files/CovidVax_DM/data/currentData02092025/included_cohort.csv.gz')

/tmp/ipykernel_1598942/857829662.py:1: DtypeWarning: Columns (11,12,13) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/mnt/dicoms/borja_files/CovidVax_DM/data/currentData02092025/included_cohort.csv.gz')


In [3]:
len(data)

7371644

In [4]:
data.shape

(7371644, 84)

In [5]:
data.columns

Index(['NIA', 'sexe', 'data_naixement', 'abs_c', 'abs', 'pais_c',
       'N_vaccine_total', 'VACUNA_1_DATA', 'VACUNA_1_MOTIU', 'VACUNA_2_DATA',
       'VACUNA_2_MOTIU', 'VACUNA_3_DATA', 'VACUNA_3_MOTIU', 'VACUNA_1_DATA_pp',
       'VACUNA_2_DATA_pp', 'VACUNA_3_DATA_pp', 'DATA_DM_min', 'DM',
       'covid_bef_vax', 'test_date', 'test_res', 'test_date_covid_2',
       'test_res_covid_2', 'test_date_covid_3', 'test_res_covid_3',
       'test_date_imc_1', 'test_res_imc_1', 'test_date_imc_2',
       'test_res_imc_2', 'test_date_imc_3', 'test_res_imc_3', 'test_date_sp_1',
       'test_res_sp_1', 'test_date_sp_2', 'test_res_sp_2', 'test_date_sp_3',
       'test_res_sp_3', 'test_date_dp_1', 'test_res_dp_1', 'test_date_dp_2',
       'test_res_dp_2', 'test_date_dp_3', 'test_res_dp_3', 'test_date_abdo_1',
       'test_res_abdo_1', 'test_date_abdo_2', 'test_res_abdo_2',
       'test_date_abdo_3', 'test_res_abdo_3', 'test_date_bg_1',
       'test_res_bg_1', 'test_date_bg_2', 'test_res_bg_2', 'test_

In [6]:
data.columns = ['NIA', 'sexe', 'data_naixement', 'abs_c', 'abs', 'pais_c',
       'N_vaccine_total', 'VACUNA_1_DATA', 'VACUNA_1_MOTIU', 'VACUNA_2_DATA',
       'VACUNA_2_MOTIU', 'VACUNA_3_DATA', 'VACUNA_3_MOTIU', 'VACUNA_1_DATA_pp',
       'VACUNA_2_DATA_pp', 'VACUNA_3_DATA_pp', 'DATA_DM_min', 'DM',
       'covid_bef_vax', 'test_date_covid_1', 'test_res_covid_1', 'test_date_covid_2',
       'test_res_covid_2', 'test_date_covid_3', 'test_res_covid_3',
       'test_date_imc_1', 'test_res_imc_1', 'test_date_imc_2',
       'test_res_imc_2', 'test_date_imc_3', 'test_res_imc_3', 'test_date_sp_1',
       'test_res_sp_1', 'test_date_sp_2', 'test_res_sp_2', 'test_date_sp_3',
       'test_res_sp_3', 'test_date_dp_1', 'test_res_dp_1', 'test_date_dp_2',
       'test_res_dp_2', 'test_date_dp_3', 'test_res_dp_3', 'test_date_abdo_1',
       'test_res_abdo_1', 'test_date_abdo_2', 'test_res_abdo_2',
       'test_date_abdo_3', 'test_res_abdo_3', 'test_date_bg_1',
       'test_res_bg_1', 'test_date_bg_2', 'test_res_bg_2', 'test_date_bg_3',
       'test_res_bg_3', 'test_date_chol_1', 'test_res_chol_1',
       'test_date_chol_2', 'test_res_chol_2', 'test_date_chol_3',
       'test_res_chol_3', 'test_date_smoking_1', 'test_res_smoking_1',
       'test_date_smoking_2', 'test_res_smoking_2', 'test_date_smoking_3',
       'test_res_smoking_3', 'test_date_gma_1', 'test_res_gma_1',
       'test_date_gma_2', 'test_res_gma_2', 'test_date_gma_3',
       'test_res_gma_3', 'test_date_sociostat_1', 'test_res_sociostat_1',
       'test_date_sociostat_2', 'test_res_sociostat_2',
       'test_date_sociostat_3', 'test_res_sociostat_3', 'age_1', 'age_2',
       'age_3', 'idabs', 'ISC reescalat']

In [7]:
# Create useful lists

In [8]:
time_covars = ['test_date_covid_1', 'test_res_covid_1', 'test_date_covid_2',
'test_res_covid_2', 'test_date_covid_3', 'test_res_covid_3',
'test_date_imc_1', 'test_res_imc_1', 'test_date_imc_2',
'test_res_imc_2', 'test_date_imc_3', 'test_res_imc_3', 'test_date_sp_1',
'test_res_sp_1', 'test_date_sp_2', 'test_res_sp_2', 'test_date_sp_3',
'test_res_sp_3', 'test_date_dp_1', 'test_res_dp_1', 'test_date_dp_2',
'test_res_dp_2', 'test_date_dp_3', 'test_res_dp_3', 'test_date_abdo_1',
'test_res_abdo_1', 'test_date_abdo_2', 'test_res_abdo_2',
'test_date_abdo_3', 'test_res_abdo_3', 'test_date_bg_1',
'test_res_bg_1', 'test_date_bg_2', 'test_res_bg_2', 'test_date_bg_3',
'test_res_bg_3', 'test_date_chol_1', 'test_res_chol_1',
'test_date_chol_2', 'test_res_chol_2', 'test_date_chol_3',
'test_res_chol_3', 'test_date_smoking_1', 'test_res_smoking_1',
'test_date_smoking_2', 'test_res_smoking_2', 'test_date_smoking_3',
'test_res_smoking_3', 'test_date_gma_1', 'test_res_gma_1',
'test_date_gma_2', 'test_res_gma_2', 'test_date_gma_3',
'test_res_gma_3', 'test_date_sociostat_1', 'test_res_sociostat_1',
'test_date_sociostat_2', 'test_res_sociostat_2',
'test_date_sociostat_3', 'test_res_sociostat_3']

In [9]:
tests = list(set([x.split('_')[2] for x in time_covars]))

In [10]:
# Preprocess

In [11]:
# Format

In [12]:
data.sexe = data.sexe.map({'H':0, 'D':1})

In [13]:
data.test_res_covid_1 = data.test_res_covid_1.map({'Positiu':'Positiu', 'Negatiu': 'Negatiu', 'Probable': 'Positiu', 'No valorat': 'Negatiu'})
data.test_res_covid_1 = data.test_res_covid_1.map({'Positiu': int(1), 'Negatiu': int(0)})

data.test_res_covid_2 = data.test_res_covid_2.map({'Positiu':'Positiu', 'Negatiu': 'Negatiu', 'Probable': 'Positiu', 'No valorat': 'Negatiu'})
data.test_res_covid_2 = data.test_res_covid_2.map({'Positiu': int(1), 'Negatiu': int(0)})

data.test_res_covid_3 = data.test_res_covid_3.map({'Positiu':'Positiu', 'Negatiu': 'Negatiu', 'Probable': 'Positiu', 'No valorat': 'Negatiu'})
data.test_res_covid_3 = data.test_res_covid_3.map({'Positiu': int(1), 'Negatiu': int(0)})

In [14]:
data['Vacuna_1'] = np.zeros(len(data))
data['Vacuna_2'] = np.zeros(len(data))
data['Vacuna_3'] = np.zeros(len(data))

data.loc[~data.VACUNA_1_DATA.isna(), 'Vacuna_1'] = int(1)
data.loc[~data.VACUNA_2_DATA.isna(), 'Vacuna_2'] = int(1)
data.loc[~data.VACUNA_3_DATA.isna(), 'Vacuna_3'] = int(1)

In [15]:
# Correct treatment "line" for those with DM date before last vax date

In [16]:
# From 3 to 2 vaxxes
data[(data.N_vaccine_total==3) & (data.DM==1) & ((data.DATA_DM_min<data.VACUNA_3_DATA)&(data.DATA_DM_min>=data.VACUNA_2_DATA))]

,NIA,sexe,data_naixement,abs_c,abs,pais_c,N_vaccine_total,VACUNA_1_DATA,VACUNA_1_MOTIU,VACUNA_2_DATA,...,test_date_sociostat_3,test_res_sociostat_3,age_1,age_2,age_3,idabs,ISC reescalat,Vacuna_1,Vacuna_2,Vacuna_3
481,4716253,0,1957-05-17T00:00:00.000Z,281.0,Badalona 10,NaN,3,2021-02-25T23:00:00.000Z,Personal centre sanitari,2021-03-18T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,63.824543,63.882078,64.616324,281.0,64.407607,1.0,1.0,1.0
782,6508256,1,1945-12-01T00:00:00.000Z,143.0,Malgrat de Mar,724.0,3,2021-01-07T23:00:00.000Z,Persones institucionalitzades,2021-01-28T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,75.156050,75.213584,75.862900,143.0,52.048451,1.0,1.0,1.0
1370,7258,1,1967-12-15T00:00:00.000Z,4.0,Amposta,724.0,3,2021-05-11T23:00:00.000Z,Patologia de risc,2021-06-09T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,2.0,53.443721,53.523174,53.832763,4.0,50.919598,1.0,1.0,1.0
1448,10376685,0,1955-01-18T00:00:00.000Z,222.0,Santa Coloma de Gramenet 1,156.0,3,2021-06-20T23:00:00.000Z,Grups d'edat,2021-07-11T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,66.468379,66.525913,67.040982,222.0,51.141867,1.0,1.0,1.0
1900,5900892,0,1944-12-04T00:00:00.000Z,63.0,Barcelona 8-G,NaN,3,2021-02-25T23:00:00.000Z,Patologia de risc,2021-03-18T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,76.282078,76.339612,76.832763,63.0,68.183293,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7318406,3458167,1,1940-10-28T00:00:00.000Z,54.0,Barcelona 7-D,724.0,3,2021-03-03T23:00:00.000Z,Patologia de risc,2021-03-24T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,80.402626,80.460160,81.109475,54.0,36.062020,1.0,1.0,1.0
7324870,6913710,1,1939-02-21T00:00:00.000Z,284.0,Granollers 1 Oest,724.0,3,2021-02-18T23:00:00.000Z,Grups d'edat,2021-03-11T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,82.051941,82.109475,82.794406,284.0,57.992381,1.0,1.0,1.0
7331189,4601885,0,1945-05-04T00:00:00.000Z,223.0,Santa Coloma de Gramenet 2,724.0,3,2021-04-22T23:00:00.000Z,Grups d'edat,2021-05-13T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,76.021804,76.079338,76.597146,223.0,66.072818,1.0,1.0,1.0
7336624,5578219,1,1945-02-11T00:00:00.000Z,186.0,Premià de Mar,NaN,3,2021-04-13T23:00:00.000Z,Grups d'edat,2021-05-04T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,76.221804,76.279338,76.830023,186.0,32.164343,1.0,1.0,1.0


In [17]:
nias_to_vax2_from3 = data[(data.N_vaccine_total==3) & (data.DM==1) & ((data.DATA_DM_min<data.VACUNA_3_DATA)&(data.DATA_DM_min>=data.VACUNA_2_DATA))]['NIA']

In [18]:
# From 3 to 1 vaxxes
data[(data.N_vaccine_total==3) & (data.DM==1) & ((data.DATA_DM_min<data.VACUNA_3_DATA) & (data.DATA_DM_min<data.VACUNA_2_DATA) & (data.DATA_DM_min>=data.VACUNA_1_DATA))]

,NIA,sexe,data_naixement,abs_c,abs,pais_c,N_vaccine_total,VACUNA_1_DATA,VACUNA_1_MOTIU,VACUNA_2_DATA,...,test_date_sociostat_3,test_res_sociostat_3,age_1,age_2,age_3,idabs,ISC reescalat,Vacuna_1,Vacuna_2,Vacuna_3
924,28435,0,1958-11-08T00:00:00.000Z,149.0,Martorell,724.0,3,2021-05-19T23:00:00.000Z,Grups d'edat,2021-07-21T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,62.572489,62.745091,63.649201,149.0,45.240799,1.0,1.0,1.0
2604,10092326,0,1958-10-10T00:00:00.000Z,109.0,Cornellà de Llobregat 2,504.0,3,2021-10-13T23:00:00.000Z,Grups d'edat,2021-11-22T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,63.054680,63.164269,63.731393,109.0,68.136792,1.0,1.0,1.0
3694,131611,1,1964-08-11T00:00:00.000Z,196.0,Sabadell 5,724.0,3,2021-03-01T23:00:00.000Z,Patologia de risc,2021-03-29T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,56.594406,56.671119,57.125913,196.0,41.999450,1.0,1.0,1.0
4544,7422987,1,1969-05-16T00:00:00.000Z,142.0,Lloret de Mar,NaN,3,2021-05-18T23:00:00.000Z,Grups d'edat,2021-06-08T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,52.043721,52.101256,52.720434,142.0,43.224473,1.0,1.0,1.0
4961,6251314,0,1960-02-06T00:00:00.000Z,76.0,Barcelona 10-F,724.0,3,2021-03-01T23:00:00.000Z,Personal centre sanitari,2021-03-22T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,1.0,61.109475,61.167009,61.665639,76.0,26.874364,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7087120,5813676,0,1981-08-21T00:00:00.000Z,2.0,Alcarràs,724.0,3,2021-03-29T23:00:00.000Z,Patologia de risc,2021-05-11T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,39.632763,39.750571,40.399886,2.0,49.502202,1.0,1.0,1.0
7093068,747570,0,1951-11-12T00:00:00.000Z,247.0,Terrassa A,724.0,3,2021-05-26T23:00:00.000Z,Grups d'edat,2021-12-13T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,69.586187,70.136872,71.005365,247.0,27.342183,1.0,1.0,1.0
7099021,19134443,1,1959-10-09T00:00:00.000Z,104.0,Cerdanyola del Vallès 1,724.0,3,2021-05-20T23:00:00.000Z,Grups d'edat,2021-07-19T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,61.657420,61.821804,62.884817,104.0,29.793668,1.0,1.0,1.0
7177299,4002875,1,1952-04-12T00:00:00.000Z,285.0,Granollers 2 Nord,724.0,3,2021-05-02T23:00:00.000Z,Grups d'edat,2021-07-01T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,69.103995,69.268379,69.495776,285.0,44.274294,1.0,1.0,1.0


In [19]:
nias_to_vax1_from3 = data[(data.N_vaccine_total==3) & (data.DM==1) & ((data.DATA_DM_min<data.VACUNA_3_DATA) & (data.DATA_DM_min<data.VACUNA_2_DATA) & (data.DATA_DM_min>=data.VACUNA_1_DATA))]['NIA']

In [20]:
# From 3 o 0 vaxxes
data[(data.N_vaccine_total==3) & (data.DM==1) & (data.DATA_DM_min<data.VACUNA_1_DATA)]

,NIA,sexe,data_naixement,abs_c,abs,pais_c,N_vaccine_total,VACUNA_1_DATA,VACUNA_1_MOTIU,VACUNA_2_DATA,...,test_date_sociostat_3,test_res_sociostat_3,age_1,age_2,age_3,idabs,ISC reescalat,Vacuna_1,Vacuna_2,Vacuna_3
222,5876073,1,1975-10-09T00:00:00.000Z,224.0,Santa Coloma de Gramenet 3,724.0,3,2021-06-07T23:00:00.000Z,Patologia de risc,2021-07-05T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,45.695776,45.772489,46.298516,224.0,60.084877,1.0,1.0,1.0
255,2627012,0,1970-01-23T00:00:00.000Z,248.0,Terrassa B,724.0,3,2021-05-20T23:00:00.000Z,Grups d'edat,2021-06-10T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,51.358790,51.416324,51.994406,248.0,70.236305,1.0,1.0,1.0
840,2949709,1,1958-10-19T00:00:00.000Z,392.0,Polinyà - Sentmenat,724.0,3,2021-03-28T23:00:00.000Z,Grups d'edat,2021-06-22T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,62.484817,62.720434,63.191667,392.0,43.766425,1.0,1.0,1.0
1100,4560596,1,1937-03-09T00:00:00.000Z,281.0,Badalona 10,724.0,3,2021-03-29T23:00:00.000Z,Grups d'edat,2021-04-19T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,84.114954,84.172489,84.890297,281.0,64.407607,1.0,1.0,1.0
1310,3349598,0,1955-02-07T00:00:00.000Z,65.0,Barcelona 9-A,724.0,3,2021-03-31T23:00:00.000Z,Patologia de risc,2021-04-28T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,2.0,66.191667,66.268379,66.693037,65.0,33.804476,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7238221,6896790,0,1957-05-21T00:00:00.000Z,384.0,Cerdanyola - Ripollet,724.0,3,2021-07-25T23:00:00.000Z,Grups d'edat,2021-08-15T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,2.0,64.224543,64.282078,65.671119,384.0,42.295399,1.0,1.0,1.0
7239428,2363432,0,1968-11-30T00:00:00.000Z,74.0,Barcelona 10-D,NaN,3,2021-05-18T23:00:00.000Z,Grups d'edat,2021-06-12T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,2.0,52.501256,52.569749,53.150571,74.0,59.182157,1.0,1.0,1.0
7251068,7417246,1,1962-03-01T00:00:00.000Z,353.0,Vilassar de Dalt,250.0,3,2021-05-11T23:00:00.000Z,Grups d'edat,2021-06-08T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,59.238242,59.314954,59.906735,353.0,18.153881,1.0,1.0,1.0
7258808,2200228,0,1971-08-02T00:00:00.000Z,28.0,Barcelona 2-H,724.0,3,2021-05-17T23:00:00.000Z,Grups d'edat,2021-07-08T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,2.0,49.827283,49.969749,50.482078,28.0,15.805092,1.0,1.0,1.0


In [21]:
nias_to_vax0_from3 = data[(data.N_vaccine_total==3) & (data.DM==1) & (data.DATA_DM_min<data.VACUNA_1_DATA)]['NIA']

In [22]:
# From 2 to 1 vaxxes
data[(data.N_vaccine_total==2) & (data.DM==1) & ((data.DATA_DM_min<data.VACUNA_2_DATA)&(data.DATA_DM_min>=data.VACUNA_1_DATA))]

,NIA,sexe,data_naixement,abs_c,abs,pais_c,N_vaccine_total,VACUNA_1_DATA,VACUNA_1_MOTIU,VACUNA_2_DATA,...,test_date_sociostat_3,test_res_sociostat_3,age_1,age_2,age_3,idabs,ISC reescalat,Vacuna_1,Vacuna_2,Vacuna_3
32284,3455951,0,1971-08-23T00:00:00.000Z,141.0,Lleida Rural 1 Nord,NaN,2,2021-05-18T23:00:00.000Z,Grups d'edat,2021-06-09T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,49.772489,49.832763,49.914954,141.0,37.102172,1.0,1.0,0.0
45556,1231232,1,1965-06-27T00:00:00.000Z,114.0,L'Escala,724.0,2,2021-05-10T23:00:00.000Z,Patologia de risc,2021-06-07T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,55.909475,55.986187,56.068379,114.0,42.605850,1.0,1.0,0.0
50653,8588216,1,1971-01-10T00:00:00.000Z,236.0,El Solsonès,724.0,2,2021-02-24T23:00:00.000Z,Altre personal essencial,2021-06-04T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,50.161530,50.435502,50.517694,236.0,45.755006,1.0,1.0,0.0
70374,462518,0,1973-05-16T00:00:00.000Z,35.0,Barcelona 3-D,724.0,2,2021-03-24T23:00:00.000Z,Altre personal essencial,2021-06-20T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,2.0,47.890297,48.131393,48.213584,35.0,35.394163,1.0,1.0,0.0
88674,82411786,0,1953-08-04T00:00:00.000Z,190.0,Sabadell 1-A,862.0,2,2021-06-14T23:00:00.000Z,Grups d'edat,2022-01-31T23:00:00.000Z,...,NaN,NaN,67.909475,68.542352,68.624543,190.0,17.274492,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7369563,10414220,1,1977-09-30T00:00:00.000Z,108.0,Cornellà de Llobregat 1,724.0,2,2021-07-19T23:00:00.000Z,Grups d'edat,2022-12-14T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,43.832763,45.238242,45.320434,108.0,40.049335,1.0,1.0,0.0
7369568,6002948,1,1955-01-18T00:00:00.000Z,225.0,Santa Coloma de Gramenet 4,NaN,2,2021-06-20T23:00:00.000Z,Grups d'edat,2022-11-14T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,66.468379,67.871119,67.953311,225.0,53.975559,1.0,1.0,0.0
7370379,12860361,0,1986-06-26T00:00:00.000Z,290.0,L'Hospitalet de Llobregat 3 Collblanc,340.0,2,2021-07-21T23:00:00.000Z,Grups d'edat,2022-11-10T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,35.095776,36.402626,36.484817,290.0,50.627094,1.0,1.0,0.0
7370892,10593050,0,1974-01-01T00:00:00.000Z,222.0,Santa Coloma de Gramenet 1,724.0,2,2021-09-03T23:00:00.000Z,Grups d'edat,2023-09-14T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,47.706735,49.736872,49.819064,222.0,51.141867,1.0,1.0,0.0


In [23]:
nias_to_vax1_from2 = data[(data.N_vaccine_total==2) & (data.DM==1) & ((data.DATA_DM_min<data.VACUNA_2_DATA)&(data.DATA_DM_min>=data.VACUNA_1_DATA))]['NIA']

In [24]:
# From 2 to 0 vaxxes
data[(data.N_vaccine_total==2) & (data.DM==1) & (data.DATA_DM_min<data.VACUNA_1_DATA)]

,NIA,sexe,data_naixement,abs_c,abs,pais_c,N_vaccine_total,VACUNA_1_DATA,VACUNA_1_MOTIU,VACUNA_2_DATA,...,test_date_sociostat_3,test_res_sociostat_3,age_1,age_2,age_3,idabs,ISC reescalat,Vacuna_1,Vacuna_2,Vacuna_3
9577,12974259,0,1977-04-06T00:00:00.000Z,32.0,Barcelona 3-A,586.0,2,2021-12-06T23:00:00.000Z,Grups d'edat,2022-08-03T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,44.701256,45.358790,45.440982,32.0,46.571176,1.0,1.0,0.0
15178,5570039,0,1948-08-29T00:00:00.000Z,228.0,Santa Coloma de Queralt,724.0,2,2021-05-11T23:00:00.000Z,Grups d'edat,2021-06-01T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,72.750571,72.808105,72.890297,228.0,43.966155,1.0,1.0,0.0
26001,6404427,1,1975-05-03T00:00:00.000Z,371.0,Igualada 1,724.0,2,2021-05-27T23:00:00.000Z,Grups d'edat,2021-06-18T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,46.101256,46.161530,46.243721,371.0,42.894603,1.0,1.0,0.0
32216,6034767,0,1969-07-12T00:00:00.000Z,137.0,Lleida 3 Eixample,724.0,2,2021-05-24T23:00:00.000Z,Grups d'edat,2021-07-01T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,4.0,51.903995,52.008105,52.090297,137.0,28.787179,1.0,1.0,0.0
57905,7536062,1,1957-01-10T00:00:00.000Z,276.0,Badalona 5,724.0,2,2021-05-13T23:00:00.000Z,Persones institucionalitzades,2021-06-03T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,64.383447,64.440982,64.523174,276.0,100.000000,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7367773,81034880,1,1949-02-20T00:00:00.000Z,286.0,Granollers 3 Centre Est,724.0,2,2022-01-07T23:00:00.000Z,Grups d'edat,2022-11-03T23:00:00.000Z,...,NaN,NaN,72.931393,73.753311,73.835502,286.0,29.305315,1.0,1.0,0.0
7368370,123769,1,1968-03-18T00:00:00.000Z,62.0,Barcelona 8-F,724.0,2,2021-08-19T23:00:00.000Z,Grups d'edat,2022-06-26T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,53.460160,54.312215,54.394406,62.0,48.620696,1.0,1.0,0.0
7368626,7190131,0,1965-09-04T00:00:00.000Z,276.0,Badalona 5,NaN,2,2021-12-21T23:00:00.000Z,Grups d'edat,2022-09-23T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,2.0,56.336872,57.093037,57.175228,276.0,100.000000,1.0,1.0,0.0
7369864,1384728,0,1956-08-08T00:00:00.000Z,351.0,Vic 2 Sud,724.0,2,2021-08-26T23:00:00.000Z,Grups d'edat,2022-06-20T23:00:00.000Z,...,2021-01-01T00:00:00.000Z,3.0,65.095776,65.912215,65.994406,351.0,46.559923,1.0,1.0,0.0


In [25]:
nias_to_vax0_from2 = data[(data.N_vaccine_total==2) & (data.DM==1) & (data.DATA_DM_min<data.VACUNA_1_DATA)]['NIA']

In [26]:
# From 1 to 0 vaxxes
data[(data.N_vaccine_total==1) & (data.DM==1) & (data.DATA_DM_min<data.VACUNA_1_DATA)]

,NIA,sexe,data_naixement,abs_c,abs,pais_c,N_vaccine_total,VACUNA_1_DATA,VACUNA_1_MOTIU,VACUNA_2_DATA,...,test_date_sociostat_3,test_res_sociostat_3,age_1,age_2,age_3,idabs,ISC reescalat,Vacuna_1,Vacuna_2,Vacuna_3
22941,5538855,0,1959-10-04T00:00:00.000Z,20.0,Barcelona 1-E,724.0,1,2021-05-18T23:00:00.000Z,Grups d'edat,NaN,...,2021-01-01T00:00:00.000Z,4.0,61.665639,61.747831,61.830023,20.0,49.422524,1.0,0.0,0.0
95154,11196582,1,2010-12-09T00:00:00.000Z,69.0,Barcelona 9-E,NaN,1,2022-01-11T23:00:00.000Z,Grups d'edat,NaN,...,2021-01-01T00:00:00.000Z,2.0,11.101256,11.183447,11.265639,69.0,56.274057,1.0,0.0,0.0
141097,1420817,0,1937-05-03T00:00:00.000Z,65.0,Barcelona 9-A,724.0,1,2021-05-10T23:00:00.000Z,Grups d'edat,NaN,...,2021-01-01T00:00:00.000Z,3.0,84.079338,84.161530,84.243721,65.0,33.804476,1.0,0.0,0.0
143944,3639736,0,1962-01-01T00:00:00.000Z,183.0,El Prat de Llobregat 2,724.0,1,2021-05-11T23:00:00.000Z,Grups d'edat,NaN,...,2021-01-01T00:00:00.000Z,2.0,59.399886,59.482078,59.564269,183.0,49.671826,1.0,0.0,0.0
261344,8614531,1,1954-06-15T00:00:00.000Z,304.0,Reus 2,724.0,1,2021-05-30T23:00:00.000Z,Grups d'edat,NaN,...,2021-01-01T00:00:00.000Z,3.0,67.005365,67.087557,67.169749,304.0,42.341641,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7360573,83552448,1,1972-01-01T00:00:00.000Z,222.0,Santa Coloma de Gramenet 1,586.0,1,2023-02-02T23:00:00.000Z,Grups d'edat,NaN,...,NaN,NaN,51.125913,51.208105,51.290297,222.0,51.141867,1.0,0.0,0.0
7365624,1607405,0,1934-03-11T00:00:00.000Z,336.0,Tordera,724.0,1,2022-10-27T23:00:00.000Z,Grups d'edat,NaN,...,NaN,NaN,88.693037,88.775228,88.857420,336.0,55.537065,1.0,0.0,0.0
7369945,83681053,0,1957-02-23T00:00:00.000Z,251.0,Terrassa E,724.0,1,2023-04-02T23:00:00.000Z,Grups d'edat,NaN,...,NaN,NaN,66.150571,66.232763,66.314954,251.0,35.356652,1.0,0.0,0.0
7371092,3983237,0,1959-08-03T00:00:00.000Z,340.0,Vila-seca,724.0,1,2023-01-04T23:00:00.000Z,Grups d'edat,NaN,...,2021-01-01T00:00:00.000Z,2.0,63.468379,63.550571,63.632763,340.0,49.457626,1.0,0.0,0.0


In [27]:
nias_to_vax0_from1 = data[(data.N_vaccine_total==1) & (data.DM==1) & (data.DATA_DM_min<data.VACUNA_1_DATA)]['NIA']

In [28]:
# Update lists
nias_to_vax2 = nias_to_vax2_from3
nias_to_vax1 = pd.Series(list(set(nias_to_vax1_from2) | set(nias_to_vax1_from3)))
nias_to_vax0 = pd.Series(list(set(nias_to_vax0_from1) | set(nias_to_vax0_from2) | set(nias_to_vax0_from3)))

In [29]:
# Correct the required information
data.loc[data.NIA.isin(nias_to_vax0), 'N_vaccine_total'] = 0
data.loc[data.NIA.isin(nias_to_vax0), 'Vacuna_1'] = int(0)
data.loc[data.NIA.isin(nias_to_vax0), 'VACUNA_1_DATA'] = np.NaN
data.loc[data.NIA.isin(nias_to_vax0), 'Vacuna_2'] = int(0)
data.loc[data.NIA.isin(nias_to_vax0), 'VACUNA_2_DATA'] = np.NaN
data.loc[data.NIA.isin(nias_to_vax0), 'Vacuna_3'] = int(0)
data.loc[data.NIA.isin(nias_to_vax0), 'VACUNA_3_DATA'] = np.NaN


In [30]:
# Correct the required information
data.loc[data.NIA.isin(nias_to_vax1), 'N_vaccine_total'] = 1
data.loc[data.NIA.isin(nias_to_vax1), 'Vacuna_2'] = int(0)
data.loc[data.NIA.isin(nias_to_vax1), 'VACUNA_2_DATA'] = np.NaN
data.loc[data.NIA.isin(nias_to_vax1), 'Vacuna_3'] = int(0)
data.loc[data.NIA.isin(nias_to_vax1), 'VACUNA_3_DATA'] = np.NaN


In [31]:
# Correct the required information
data.loc[data.NIA.isin(nias_to_vax2), 'N_vaccine_total'] = 2
data.loc[data.NIA.isin(nias_to_vax2), 'Vacuna_3'] = int(0)
data.loc[data.NIA.isin(nias_to_vax2), 'VACUNA_3_DATA'] = np.NaN


In [32]:
# Date formatting

In [33]:
data.data_naixement = pd.to_datetime(data.data_naixement)

In [34]:
data.data_naixement = data.data_naixement.dt.tz_convert(None)

In [35]:
## RECOMMENDED VACCINE TIME for those WITH vaccine. Information taken from spanish health ministry
recomm_date_1vax_above50 = pd.to_datetime('2021-02-01')
recomm_date_2vax_above50 = pd.to_datetime('2021-03-01')
recomm_date_3vax_above50 = pd.to_datetime('2021-04-01')

recomm_date_1vax_below50 = pd.to_datetime('2021-06-01')
recomm_date_2vax_below50 = pd.to_datetime('2021-07-01')
recomm_date_3vax_below50 = pd.to_datetime('2021-08-01')

In [36]:
# Set 'per protocol' vaccination dates for those without that info

#  above 50
data.loc[((data.VACUNA_1_DATA_pp.isna())&(((recomm_date_1vax_above50-data.data_naixement).values/pd.Timedelta('365 days'))>49.)), 'VACUNA_1_DATA_pp'] = pd.to_datetime(recomm_date_1vax_above50)

data.loc[((data.VACUNA_2_DATA_pp.isna())&(((recomm_date_1vax_above50-data.data_naixement).values/pd.Timedelta('365 days'))>49.)), 'VACUNA_2_DATA_pp'] = pd.to_datetime(recomm_date_2vax_above50)

data.loc[((data.VACUNA_3_DATA_pp.isna())&(((recomm_date_1vax_above50-data.data_naixement).values/pd.Timedelta('365 days'))>49.)), 'VACUNA_3_DATA_pp'] = pd.to_datetime(recomm_date_3vax_above50)

#  below 50
data.loc[((data.VACUNA_1_DATA_pp.isna())&(((recomm_date_1vax_below50-data.data_naixement).values/pd.Timedelta('365 days'))<=50.)), 'VACUNA_1_DATA_pp'] = pd.to_datetime(recomm_date_1vax_below50)

data.loc[((data.VACUNA_2_DATA_pp.isna())&(((recomm_date_1vax_below50-data.data_naixement).values/pd.Timedelta('365 days'))<=50.)), 'VACUNA_2_DATA_pp'] = pd.to_datetime(recomm_date_2vax_below50)

data.loc[((data.VACUNA_3_DATA_pp.isna())&(((recomm_date_1vax_below50-data.data_naixement).values/pd.Timedelta('365 days'))<=50.)), 'VACUNA_3_DATA_pp'] = pd.to_datetime(recomm_date_3vax_below50)


In [37]:
### Check NaNs

In [38]:
percent_missing = data.isnull().sum() * 100 / len(data)
missing_value_df = pd.DataFrame({'column_name': data.columns,'percent_missing': percent_missing})
missing_value_df.sort_values('percent_missing', inplace=True)


In [39]:
missing_value_df.head(50)

,column_name,percent_missing
NIA,NIA,0.000000
Vacuna_1,Vacuna_1,0.000000
age_3,age_3,0.000000
age_2,age_2,0.000000
age_1,age_1,0.000000
Vacuna_2,Vacuna_2,0.000000
covid_bef_vax,covid_bef_vax,0.000000
DM,DM,0.000000
VACUNA_3_DATA_pp,VACUNA_3_DATA_pp,0.000000
VACUNA_2_DATA_pp,VACUNA_2_DATA_pp,0.000000


In [40]:
missing_value_df.tail(50)

,column_name,percent_missing
test_res_dp_3,test_res_dp_3,27.705502
test_date_sp_2,test_date_sp_2,28.126236
test_res_sp_2,test_res_sp_2,28.128692
test_date_dp_2,test_date_dp_2,28.145092
test_res_dp_2,test_res_dp_2,28.147588
test_date_sp_1,test_date_sp_1,28.392269
test_res_sp_1,test_res_sp_1,28.394779
test_date_dp_1,test_date_dp_1,28.411003
test_res_dp_1,test_res_dp_1,28.413580
VACUNA_2_MOTIU,VACUNA_2_MOTIU,29.706182


In [41]:
### Check covariate variety and nans

In [42]:
### Result/date variety and NaN

In [43]:
data.shape

(7371644, 87)

In [44]:
for test_i in tests:
    cols = ['test_date_{}_1'.format(test_i), 'test_date_{}_2'.format(test_i), 'test_date_{}_3'.format(test_i)]
    
    # Count number of equal pairs (ignoring NaNs)
    eq_ab = (data[cols[0]] == data[cols[1]]) & data[cols[0]].notna() & data[cols[1]].notna()
    eq_ac = (data[cols[0]] == data[cols[2]]) & data[cols[0]].notna() & data[cols[2]].notna()
    eq_bc = (data[cols[1]] == data[cols[2]]) & data[cols[1]].notna() & data[cols[2]].notna()

    equal_pairs = eq_ab.astype(int) + eq_ac.astype(int) + eq_bc.astype(int)

    # Count non-null values per row
    non_nulls = data[cols].notna().sum(axis=1)

    # Initialize result with NaNs
    result = pd.Series(np.nan, index=data.index)

    # Assign values based on number of equal pairs
    result[equal_pairs == 3] = 3  # All three equal
    result[(equal_pairs == 1) | (equal_pairs == 2)] = 1  # At least two equal
    result[(equal_pairs == 0) & (non_nulls >= 1)] = 0  # All different
    
    #print(test_i)
    #print(len(result))
    
    data[test_i] = result

In [45]:
for test_i in tests:
    print(data[test_i].value_counts(dropna=False))

imc
3.0    4465695
NaN    2343762
1.0     500580
0.0      61607
Name: count, dtype: int64
covid
NaN    3072590
3.0    2384375
1.0    1235039
0.0     679640
Name: count, dtype: int64
abdo
NaN    6936647
3.0     414816
1.0      13038
0.0       7143
Name: count, dtype: int64
sociostat
3.0    6565990
NaN     802721
1.0       2933
Name: count, dtype: int64
bg
NaN    3879020
3.0    3489231
1.0       2660
0.0        733
Name: count, dtype: int64
smoking
3.0    4284518
NaN    2998136
1.0      54777
0.0      34213
Name: count, dtype: int64
chol
NaN    3948381
3.0    3423157
1.0         79
0.0         27
Name: count, dtype: int64
gma
3.0    4602408
1.0    2165448
NaN     586997
0.0      16791
Name: count, dtype: int64
dp
3.0    4463534
NaN    2042169
1.0     758446
0.0     107495
Name: count, dtype: int64
sp
3.0    4464666
NaN    2040808
1.0     758670
0.0     107500
Name: count, dtype: int64


In [46]:
data.shape

(7371644, 97)

In [47]:
### Outlier removal

In [48]:
# IMC above 70 becomes NaN
data.loc[data.test_res_imc_1>70, 'test_res_imc_1'] = np.NaN
data.loc[data.test_res_imc_2>70, 'test_res_imc_2'] = np.NaN
data.loc[data.test_res_imc_2>70, 'test_res_imc_2'] = np.NaN

In [49]:
data.shape

(7371644, 97)

In [50]:
# People of more than 110 years must be dead
data.data_naixement = data.data_naixement.dt.year
data = data[data.data_naixement>1915]
data.reset_index(inplace=True, drop=True)

data.data_naixement = 2024 - data.data_naixement

In [51]:
data.shape

(7371633, 97)

In [52]:
# Chol 0 or above 400 becomes NaN
data.loc[data.test_res_chol_1==0, 'test_res_chol_1'] = np.NaN
data.loc[data.test_res_chol_2==0, 'test_res_chol_2'] = np.NaN
data.loc[data.test_res_chol_2==0, 'test_res_chol_2'] = np.NaN

data.loc[data.test_res_chol_1>400, 'test_res_chol_1'] = np.NaN
data.loc[data.test_res_chol_2>400, 'test_res_chol_2'] = np.NaN
data.loc[data.test_res_chol_2>400, 'test_res_chol_2'] = np.NaN

In [53]:
data.shape

(7371633, 97)

In [54]:
# BG 0 or above 200 becomes NaN
data.loc[data.test_res_bg_1==0, 'test_res_bg_1'] = np.NaN
data.loc[data.test_res_bg_2==0, 'test_res_bg_2'] = np.NaN
data.loc[data.test_res_bg_2==0, 'test_res_bg_2'] = np.NaN

data.loc[data.test_res_bg_1>200, 'test_res_bg_1'] = np.NaN
data.loc[data.test_res_bg_2>200, 'test_res_bg_2'] = np.NaN
data.loc[data.test_res_bg_2>200, 'test_res_bg_2'] = np.NaN

In [55]:
data.shape

(7371633, 97)

In [56]:
# DP above 150 becomes NaN
data.loc[data.test_res_dp_1>150, 'test_res_dp_1'] = np.NaN
data.loc[data.test_res_dp_2>150, 'test_res_dp_2'] = np.NaN
data.loc[data.test_res_dp_2>150, 'test_res_dp_2'] = np.NaN

In [57]:
data.shape

(7371633, 97)

In [58]:
# SP above 250 becomes NaN
data.loc[data.test_res_sp_1>250, 'test_res_sp_1'] = np.NaN
data.loc[data.test_res_sp_2>250, 'test_res_sp_2'] = np.NaN
data.loc[data.test_res_sp_2>250, 'test_res_sp_2'] = np.NaN

In [59]:
data.shape

(7371633, 97)

In [60]:
# Abdominal perimeter above 250 becomes NaN
data.loc[data.test_res_abdo_1>250, 'test_res_abdo_1'] = np.NaN
data.loc[data.test_res_abdo_2>250, 'test_res_abdo_2'] = np.NaN
data.loc[data.test_res_abdo_2>250, 'test_res_abdo_2'] = np.NaN

In [61]:
data.shape

(7371633, 97)

In [62]:
### Baseline covs NaN imputation using iterative imputer. Of baseline covars, have missings: test_res_sociostat_1, abs_c, pais_c. Use iterimp with these + data_naixement + sexe.
# MAR approach

In [63]:
for col in ['data_naixement','sexe','pais_c','abs_c','test_res_sociostat_1']:
    print(data[col].value_counts())

data_naixement
48     136050
49     135604
47     135403
46     133913
50     132856
        ...  
104       109
105        57
106        42
107        14
108        10
Name: count, Length: 105, dtype: int64
sexe
1    3763102
0    3608531
Name: count, dtype: int64
pais_c
724.0    4457513
504.0     243039
170.0     115322
642.0      85266
380.0      79976
          ...   
583.0          1
316.0          1
598.0          1
10.0           1
16.0           1
Name: count, Length: 223, dtype: int64
abs_c
119.0    48213
13.0     46050
199.0    43434
251.0    43186
142.0    42907
         ...  
180.0     3431
181.0     3356
228.0     3156
129.0     2229
112.0     2093
Name: count, Length: 377, dtype: int64
test_res_sociostat_1
3.0    3185764
2.0    2516190
4.0     779385
1.0      84641
Name: count, dtype: int64


In [64]:
iter_imp = IterativeImputer(estimator = RandomForestClassifier(n_jobs=4, n_estimators=50, max_depth=20),
                            random_state=0, initial_strategy='most_frequent', 
                            skip_complete=True, imputation_order='ascending', 
                            min_value=data[['data_naixement','sexe','pais_c','abs_c','test_res_sociostat_1']].min().values, 
                            max_value=data[['data_naixement','sexe','pais_c','abs_c','test_res_sociostat_1']].max().values,
                            verbose=1)
gc.collect()

0

In [65]:
data[['data_naixement','sexe','pais_c','abs_c','test_res_sociostat_1']] = iter_imp.fit_transform(data[['data_naixement','sexe','pais_c','abs_c','test_res_sociostat_1']])

[IterativeImputer] Completing matrix with shape (7371633, 5)
[IterativeImputer] Change: 712.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 712.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 712.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 762.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 716.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 712.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 770.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 770.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 716.0, scaled tolerance: 0.894 
[IterativeImputer] Change: 762.0, scaled tolerance: 0.894 


/home/bvelasco/miniconda3/envs/python3.10/lib/python3.10/site-packages/sklearn/impute/_iterative.py:801: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


In [66]:
del iter_imp
gc.collect()

308

In [67]:
for col in ['data_naixement','sexe','pais_c','abs_c','test_res_sociostat_1']:
    print(data[col].value_counts())

data_naixement
48.0     136050
49.0     135604
47.0     135403
46.0     133913
50.0     132856
          ...  
104.0       109
105.0        57
106.0        42
107.0        14
108.0        10
Name: count, Length: 105, dtype: int64
sexe
1.0    3763102
0.0    3608531
Name: count, dtype: int64
pais_c
724.0    5888866
504.0     245313
170.0     115358
642.0      85338
380.0      80155
          ...   
583.0          1
316.0          1
598.0          1
10.0           1
16.0           1
Name: count, Length: 223, dtype: int64
abs_c
119.0    48213
13.0     46051
199.0    43434
251.0    43186
142.0    42907
         ...  
180.0     3431
181.0     3356
228.0     3156
129.0     2229
112.0     2093
Name: count, Length: 377, dtype: int64
test_res_sociostat_1
3.0    3897945
2.0    2593658
4.0     794537
1.0      85493
Name: count, dtype: int64


In [68]:
# Time-varying covars: MAR approach

In [69]:
dag_list = ['BMI -> Blood_pressure_dp',
'BMI -> Blood_pressure_sp',
'BMI -> Charlson_i',
'BMI -> Covid_infection',
'BMI -> Pre_diabetes',
'BMI -> Waist_size',
'Birth_year -> BMI',
'Birth_year -> Blood_pressure_dp',
'Birth_year -> Blood_pressure_sp',
'Birth_year -> Charlson_i',
'Birth_year -> Cholesterol',
'Birth_year -> Pre_diabetes',
'Birth_year -> Smoking',
'Birth_year -> Waist_size',
'Blood_pressure_dp -> Charlson_i',
'Blood_pressure_sp -> Charlson_i',
'Charlson_i -> Covid_infection',
'Cholesterol -> Charlson_i',
'Country_origin -> BMI',
'Country_origin -> Blood_pressure_dp',
'Country_origin -> Blood_pressure_sp',
'Country_origin -> Charlson_i',
'Country_origin -> Cholesterol',
'Country_origin -> Pre_diabetes',
'Country_origin -> Smoking',
'Country_origin -> Waist_size',
'Pre_diabetes -> Charlson_i',
'Pre_diabetes -> Covid_infection',
'Smoking -> BMI',
'Smoking -> Blood_pressure_dp',
'Smoking -> Blood_pressure_sp',
'Smoking -> Charlson_i',
'Smoking -> Covid_infection',
'Smoking -> Waist_size',
'Socio_status -> BMI',
'Socio_status -> Blood_pressure_dp',
'Socio_status -> Blood_pressure_sp',
'Socio_status -> Cholesterol',
'Socio_status -> Covid_infection',
'Socio_status -> Pre_diabetes',
'Socio_status -> Smoking']

In [70]:
pair_list = []
for pair in dag_list:
    pair_list.append(pair.split(' -> '))

In [71]:
pair_list = pd.DataFrame(pair_list).replace({'BMI': 'imc',
 'Birth_year': 'data_naixement',
 'Waist_size':  'abdo',
 'Cholesterol':  'chol',
 'Socio_status':  'sociostat',
 'Charlson_i':  'gma',
 'Blood_pressure_dp':  'dp',
 'Blood_pressure_sp':  'sp',
 'Covid_infection': 'covid',
 'Pre_diabetes': 'bg',
 'Country_origin': 'pais_c',
 'Smoking': 'smoking'}).values.tolist()

In [72]:
### Imputting with random forest strategy

In [73]:
# Test result

In [74]:
# 'Flatten' list 
flat_list = [x for inner_list in pair_list for x in inner_list]

for elem in set(flat_list)-set(['pais_c', 'data_naixement', 'sociostat']):
    print(elem)
    affecting_vars = []

    for pair in pair_list:
        if pair[1]==elem:
            affecting_vars.append(pair[0])

    affecting_vars_fullname = []
    for var in affecting_vars:
        affecting_vars_fullname.append(var)
    
    print(affecting_vars_fullname)
    
    if elem!='gma':
        njobs=36
    else:
        njobs=18
        
    if elem=='covid' or elem=='smoking':
        regr = RandomForestClassifier(n_jobs=njobs)
    else:
        regr = RandomForestRegressor(n_jobs=njobs)
    
    regr.fit(data.loc[~data['test_res_{}_1'.format(elem)].isna(), affecting_vars_fullname], data.loc[~data['test_res_{}_1'.format(elem)].isna(), 'test_res_{}_1'.format(elem)])
    data.loc[data['test_res_{}_1'.format(elem)].isna(), 'test_res_{}_1'.format(elem)] = regr.predict(data.loc[data['test_res_{}_1'.format(elem)].isna(), affecting_vars_fullname])
    gc.collect()
    print('*****')

gc.collect()

imc
['data_naixement', 'pais_c', 'smoking', 'sociostat']
*****
covid
['imc', 'gma', 'bg', 'smoking', 'sociostat']
*****
abdo
['imc', 'data_naixement', 'pais_c', 'smoking']
*****
smoking
['data_naixement', 'pais_c', 'sociostat']
*****
chol
['data_naixement', 'pais_c', 'sociostat']
*****
sp
['imc', 'data_naixement', 'pais_c', 'smoking', 'sociostat']
*****
gma
['imc', 'data_naixement', 'dp', 'sp', 'chol', 'pais_c', 'bg', 'smoking']
*****
dp
['imc', 'data_naixement', 'pais_c', 'smoking', 'sociostat']
*****
bg
['imc', 'data_naixement', 'pais_c', 'sociostat']
*****


0

In [75]:
# Test date. Input all at 2021-01-01. Is this the best approach?

In [76]:
for test in tests:
    data['test_date_{}_1'.format(test)] = data['test_date_{}_1'.format(test)].fillna(value='2021-01-01T00:00:00.000Z')

In [77]:
### Carry-forward

In [78]:
for test in tests:
    for i in range(2,4):
        data.loc[data['test_res_{}_{}'.format(test,i)].isna(), 'test_res_{}_{}'.format(test,i)] = data.loc[data['test_res_{}_{}'.format(test,i)].isna(), 'test_res_{}_{}'.format(test,i-1)]
        data.loc[data['test_date_{}_{}'.format(test,i)].isna(), 'test_date_{}_{}'.format(test,i)] = data.loc[data['test_date_{}_{}'.format(test,i)].isna(), 'test_date_{}_{}'.format(test,i-1)]

In [79]:
data.shape

(7371633, 97)

In [80]:
data.isna().any().head(50)

NIA                  False
sexe                 False
data_naixement       False
abs_c                False
abs                   True
pais_c               False
N_vaccine_total      False
VACUNA_1_DATA         True
VACUNA_1_MOTIU        True
VACUNA_2_DATA         True
VACUNA_2_MOTIU        True
VACUNA_3_DATA         True
VACUNA_3_MOTIU        True
VACUNA_1_DATA_pp     False
VACUNA_2_DATA_pp     False
VACUNA_3_DATA_pp     False
DATA_DM_min           True
DM                   False
covid_bef_vax        False
test_date_covid_1    False
test_res_covid_1     False
test_date_covid_2    False
test_res_covid_2     False
test_date_covid_3    False
test_res_covid_3     False
test_date_imc_1      False
test_res_imc_1       False
test_date_imc_2      False
test_res_imc_2       False
test_date_imc_3      False
test_res_imc_3       False
test_date_sp_1       False
test_res_sp_1        False
test_date_sp_2       False
test_res_sp_2        False
test_date_sp_3       False
test_res_sp_3        False
t

In [81]:
data.isna().any().tail(50)

test_date_abdo_3         False
test_res_abdo_3          False
test_date_bg_1           False
test_res_bg_1            False
test_date_bg_2           False
test_res_bg_2            False
test_date_bg_3           False
test_res_bg_3            False
test_date_chol_1         False
test_res_chol_1          False
test_date_chol_2         False
test_res_chol_2          False
test_date_chol_3         False
test_res_chol_3          False
test_date_smoking_1      False
test_res_smoking_1       False
test_date_smoking_2      False
test_res_smoking_2       False
test_date_smoking_3      False
test_res_smoking_3       False
test_date_gma_1          False
test_res_gma_1           False
test_date_gma_2          False
test_res_gma_2           False
test_date_gma_3          False
test_res_gma_3           False
test_date_sociostat_1    False
test_res_sociostat_1     False
test_date_sociostat_2    False
test_res_sociostat_2     False
test_date_sociostat_3    False
test_res_sociostat_3     False
age_1   

In [ ]:
# Final preprocess steps (from computing scripts)

In [ ]:
data['VACUNA_1_DATA'] = pd.to_datetime(data['VACUNA_1_DATA'])
data['VACUNA_2_DATA'] = pd.to_datetime(data['VACUNA_2_DATA'])
data['VACUNA_3_DATA'] = pd.to_datetime(data['VACUNA_3_DATA'])
data['VACUNA_1_DATA_pp'] = pd.to_datetime(data['VACUNA_1_DATA_pp'], format='mixed', utc=True)
data['VACUNA_2_DATA_pp'] = pd.to_datetime(data['VACUNA_2_DATA_pp'], format='mixed', utc=True)
data['VACUNA_3_DATA_pp'] = pd.to_datetime(data['VACUNA_3_DATA_pp'], format='mixed', utc=True)

if data.test_res_abdo_1.dtype=='O':
  data.test_res_abdo_1 = pd.factorize(data.test_res_abdo_1)[0]
  data.test_res_abdo_2 = pd.factorize(data.test_res_abdo_2)[0]
  data.test_res_abdo_3 = pd.factorize(data.test_res_abdo_3)[0]

In [ ]:
### Save

In [ ]:
data.to_csv('/mnt/dicoms/borja_files/CovidVax_DM/data/currentData02092025/included_cohort_prep_mar.csv')

In [ ]:
### Save sample

In [ ]:
data_sample = data.sample(frac=0.05, random_state=1)

In [ ]:
data_sample.to_csv('/mnt/dicoms/borja_files/CovidVax_DM/data/currentData02092025/included_cohort_sample_prep_mar.csv')

In [90]:
i = 0
# Loop over chunks
for data_i in np.array_split(data, 100):
    i+=1
    print(i)
    data_i.to_csv('/mnt/dicoms/borja_files/CovidVax_DM/data/currentData02092025/chunks_100_mar/included_cohort_prep_mar_{}.csv'.format(i))

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
